In [191]:
import pandas as pd
import json
import openpyxl

In [192]:
asset_file = "/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/DPP_FIBROTOR_ER15_V2.xlsx"

wb = openpyxl.load_workbook(asset_file, data_only=True)
wb_names = wb.sheetnames
print(wb.sheetnames)

['Digital Nameplate', 'Handover Documentation', 'Technical Data', 'Carbon Footprint', 'Maintenance Instructions', 'Sheet1']


In [193]:
with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA 02006-3-0-1_Template_Digital Nameplate.json") as f:
    digital_nameplate = json.load(f)
    
with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA 02004-2-0_Template_HandoverDocumentation.json") as f:
    handover_documentation = json.load(f)

with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA 02003_Template_TechnicalData.json") as f:
    technical_data = json.load(f)

with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA 02023 _Template_CarbonFootprint.json") as f:
    carbon_footprint = json.load(f)

with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA_02018_Template_MaintenanceInstructions.json") as f:
    maintenance_instructions = json.load(f)

In [194]:
from datetime import datetime, timedelta
def coerce_value(value, value_type: str) -> str:
    """
    Coerces Excel-parsed values into AAS-compliant string representations 
    based on the target element's valueType (e.g., xs:boolean, xs:integer).
    """
    if value is None or str(value).strip() in ("", "N/A", "__MISSING__"):
        return None
    
    val_str = str(value).strip()
    value_type_lower = value_type.lower()
    if "boolean" in value_type_lower:
        if val_str.lower() in ("true", "1", "yes", "y", "true.0"):
            return "true"
        return "false"
        
    elif any(x in value_type_lower for x in ("integer", "int", "long", "short")):
        try:
            return str(int(float(val_str)))
        except ValueError:
            return val_str
    elif "date" in value_type_lower:
        print("the value is type date")
        try:
            return datetime.strptime(val_str, "%Y-%m-%d").strftime("%Y-%m-%d")
        except ValueError:
            pass
        try:
            return datetime.strptime(val_str, "%d/%m/%Y").strftime("%Y-%m-%d")
        except ValueError:
            pass
        try:
            serial = float(val_str)
            excel_epoch = datetime(1899, 12, 30)
            return (excel_epoch + timedelta(days=serial)).strftime("%Y-%m-%d")
        except ValueError:
            pass
        return val_str
    elif any(x in value_type_lower for x in ("double", "float", "decimal")):
        try:
            return str(float(val_str))
        except ValueError:
            return val_str
            
    return val_str

In [195]:
def extract_table(rows, header_row_idx, key_idx, val_idx, ob_idx, sheet_name, missing_mandatory_log):
    """
    Extracts key-value pairs from a table using pre-located column indexes.
    Returns the extracted data and the index of the row where extraction stopped.
    """
    # print(rows[header_row_idx], key_idx, val_idx, ob_idx, sheet_name)
    data = {}
    row_idx = header_row_idx + 1  # Start reading data from the row after the header
    
    while row_idx < len(rows):
        row = rows[row_idx]
        
        # Stop if the row is empty or doesn't have a key cell
        if not row or len(row) <= key_idx:
            break
            
        key = row[key_idx]
        # print("the key is", key, row)
        if key is None or str(key).strip() == "":
            break  # End of this table
            
        key = str(key).strip()
        
        # Retrieve values using determined indices safely
        value = row[val_idx] if val_idx is not None and val_idx < len(row) else None
        obligation = row[ob_idx] if ob_idx is not None and ob_idx < len(row) else None
        # print("the value is", value,"the index of the value is", val_idx, "the obligation is", obligation)
        if isinstance(value, str):
            value = value.strip()

        is_empty = value is None or str(value).strip() in ("", "N/A")
        
        if obligation == "Mandatory" and is_empty:
            data[key] = "__MISSING__"
            missing_mandatory_log.append(f"{sheet_name} -> {key}")
        else:
            data[key] = None if is_empty else value
            
        row_idx += 1
        
    return data, row_idx

In [196]:
def find_submodels(sheet, missing_mandatory_log):
    """
    Scans the sheet for tables starting with a header row containing 'idShort'.
    Extracts them into a scoped dictionary.
    """
    submodels = {}
    rows = list(sheet.iter_rows(values_only=True))
    
    i = 0
    while i < len(rows):
        row = rows[i]
        if not row:
            i += 1
            continue
        
        # Look for the presence of "idShort" or "Element Name (idShort)" in the row
        key_idx = next((idx for idx, cell in enumerate(row) if cell and "Element Name (idShort)" in str(cell)), None)
        
        if key_idx is not None:
            # Table/Submodel name is typically in the row directly above (i-1)
            name = "Unknown"
            if i > 0:
                prev_row = rows[i-1]
                if prev_row:
                    # Safely grab the first non-empty text in the row above as the table name
                    name_cell = next((cell for cell in prev_row if cell is not None and str(cell).strip() != ""), "Unknown")
                    name = str(name_cell).strip()
            
            # Dynamically locate "Value" and "Obligation" columns in this header row
            val_idx = next((idx for idx, cell in enumerate(row) if cell and str(cell).strip() == "Actual Value"), None)
            ob_idx = next((idx for idx, cell in enumerate(row) if cell and str(cell).strip() == "Obligation"), None)
            
            # Extract this table's data and get the index of where the table ends
            table_data, end_row_idx = extract_table(
                rows=rows,
                header_row_idx=i,
                key_idx=key_idx,
                val_idx=val_idx,
                ob_idx=ob_idx,
                sheet_name=sheet.title,
                missing_mandatory_log=missing_mandatory_log
            )
            
            submodels[name] = table_data
            i = end_row_idx  # Resume scanning the sheet *after* this table
        else:
            i += 1
            
    return submodels

In [197]:
def fill_template(elements: list, company_data: dict, path: tuple = ()) -> None:
    """
    Recursively fills the AAS JSON template elements using sheet-scoped data.
    Automatically applies 'coerce_value' for proper AAS type compatibility.
    """
    for element in elements:
        if not isinstance(element, dict):
            continue
            
        model = element.get("modelType")
        id_short = element.get("idShort")
        if not id_short:
            continue
            
        current_path = path + (id_short,)
        path_str = ".".join(current_path)
        
        # Prioritize hierarchically nested path keys, fallback to flat idShort
        value = company_data.get(path_str, company_data.get(id_short))

        if model == "Property":
            if value is not None and value != "__MISSING__":
                val_type = element.get("valueType", "xs:string")
                # Here is where coerce_value is called to check/transform the data type!
                element["value"] = coerce_value(value, val_type)
            elif value == "__MISSING__":
                element["value"] = "__MISSING__"

        elif model == "MultiLanguageProperty":
            if value is not None:
                if "value" not in element or not isinstance(element["value"], list):
                    element["value"] = []
                
                if len(element["value"]) == 0:
                    element["value"].append({"language": "en", "text": ""})
                
                element["value"][0]["text"] = str(value)
        elif model == "File":
                # In AAS, "value" of a File represents its path or URI.
                # You might also want to dynamically set "contentType" (e.g. "image/png")
                element["value"] = str(value) if value is not None else ""

            # 4. Range Property (e.g. if company_data contains a tuple/dict)
        elif model == "Range":
                if isinstance(value, dict):
                    element["min"] = str(value.get("min", ""))
                    element["max"] = str(value.get("max", ""))
                elif isinstance(value, (list, tuple)) and len(value) == 2:
                    element["min"] = str(value[0])
                    element["max"] = str(value[1])

        elif model in ("SubmodelElementCollection", "SubmodelElementList"):
            nested_elements = element.get("value", [])
            if isinstance(nested_elements, list):
                fill_template(nested_elements, company_data, current_path)

In [198]:
import json

# Setup template mappings to Excel sheet names
template_registry = { 
    "Digital Nameplate": "IDTA 02006-3-0-1_Template_Digital Nameplate.json",
    "Handover Documentation": "IDTA 02004-2-0_Template_HandoverDocumentation.json",
    "Technical Data": "IDTA 02003_Template_TechnicalData.json",
    "Carbon Footprint": "IDTA 02023 _Template_CarbonFootprint.json",
    "Maintenance Instructions": "IDTA_02018_Template_MaintenanceInstructions.json"
}


# Tracking validation errors across the entire pipeline
all_missing_mandatory = []

# Process each sheet individually to protect namespace boundaries
for sheet_name, template_path in template_registry.items():
    if sheet_name not in wb.sheetnames:
        print(f"Skipping {sheet_name}: Not found in workbook.")
        continue
        
    sheet = wb[sheet_name]
    
    # 1. Discover and extract submodels on this specific sheet
    # Pass all_missing_mandatory so find_submodels can log empty mandatory fields directly
    submodels = find_submodels(sheet, all_missing_mandatory) 
    
    # 2. Scope and flatten the extracted tables specifically to this sheet
    sheet_data = {}
    for table_name, table_data in submodels.items():
        sheet_data.update(table_data)
        
    # 3. Load the corresponding IDTA JSON Template
    try:
        with open(template_path, "r", encoding="utf-8") as f:
            template_json = json.load(f)
    except FileNotFoundError:
        print(f"Error: Template file not found -> {template_path}")
        continue
        
    # 4. Fill the template recursively using only this sheet's scoped data
    if "submodels" in template_json and len(template_json["submodels"]) > 0:
        submodel_elements = template_json["submodels"][0].get("submodelElements", [])
        fill_template(submodel_elements, sheet_data)
        
    # 5. Save the output
    output_filename = f"output_{sheet_name.replace(' ', '_').lower()}.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(template_json, f, indent=4, ensure_ascii=False)
    print(f"Successfully generated: {output_filename}")

# Report validation issues
if all_missing_mandatory:
    print(f"\nValidation Warning: {len(all_missing_mandatory)} mandatory fields are missing:")
    for item in all_missing_mandatory:
        print(f"  - {item}")
else:
    print("\nAll mandatory fields successfully populated!")

the value is type date
Successfully generated: output_digital_nameplate.json
Successfully generated: output_handover_documentation.json
Successfully generated: output_technical_data.json
Successfully generated: output_carbon_footprint.json
Successfully generated: output_maintenance_instructions.json

Validation Warning: 19 mandatory fields are missing:
  - Handover Documentation -> DocumentIds
  - Handover Documentation -> DocumentClassifications
  - Handover Documentation -> DocumentVersions
  - Handover Documentation -> Language
  - Handover Documentation -> DigitalFiles
  - Handover Documentation -> Language
  - Handover Documentation -> DigitalFiles
  - Technical Data -> ValidDate
  - Carbon Footprint -> LifeCyclePhases
  - Carbon Footprint -> GoodsHandoverAddress
  - Carbon Footprint -> PcfCalculationMethod
  - Carbon Footprint -> ProductOrSectorSpecificRule
  - Carbon Footprint -> PcfInformation
  - Carbon Footprint -> PcfRuleOperator
  - Carbon Footprint -> PcfRuleName
  - Carbo

In [199]:
import json

# 2. Load your raw Submodel content files (the ones you got from Source 7 & 8)
with open("output_carbon_footprint.json", "r") as f:
    carbon_footprint_elements = json.load(f)

with open("output_technical_data.json", "r") as f:
    technical_data_elements = json.load(f)
    
with open("output_maintenance_instructions.json", "r") as f:
    maintenance_instructions_elements = json.load(f)
    
with open("output_digital_nameplate.json", "r") as f:
    nameplate_elements = json.load(f)
    
with open("output_handover_documentation.json", "r") as f:
    handover_documentation_elements = json.load(f)

print(nameplate_elements["submodels"][0]["submodelElements"][0]["value"])
uri_of_the_product = nameplate_elements["submodels"][0]["submodelElements"][0]["value"]
# print(f"Using URI of the product: {uri_of_the_product}")
# 3. Construct the complete AAS Environment using the URI
aas_environment = {
    "assetAdministrationShells": [
        {
            "id": f"{uri_of_the_product}/aas",
            "idShort": "AssetAdministrationShell",
            "assetInformation": {
                "assetKind": "Instance",
                "globalAssetId": uri_of_the_product
            },
            "submodels": [
                {
                    "type": "ModelReference",
                    "keys": [
                        {
                            "type": "Submodel",
                            "value": f"{uri_of_the_product}/submodels/carbon-footprint"
                        }
                    ]
                },
                {
                    "type": "ModelReference",
                    "keys": [
                        {
                            "type": "Submodel",
                            "value": f"{uri_of_the_product}/submodels/technical-data"
                        }
                    ]
                },
                {
                    "type": "ModelReference",
                    "keys": [
                        {
                            "type": "Submodel",
                            "value": f"{uri_of_the_product}/submodels/digital-nameplate"
                        }
                    ]
                },
                {
                    "type": "ModelReference",
                    "keys": [
                        {
                            "type": "Submodel",
                            "value": f"{uri_of_the_product}/submodels/handover-documentation"
                        }
                    ]
                },
                {
                    "type": "ModelReference",
                    "keys": [
                        {
                            "type": "Submodel",
                            "value": f"{uri_of_the_product}/submodels/maintenance-instructions"
                        }
                    ]
                },
            ]
        }
    ],
    "submodels": [
        {
            "id": f"{uri_of_the_product}/submodels/carbon-footprint",
            "idShort": "CarbonFootprint",
            "submodelElements": carbon_footprint_elements
        },
        {
            "id": f"{uri_of_the_product}/submodels/technical-data",
            "idShort": "TechnicalData",
            "submodelElements": technical_data_elements
        },
        {
            "id": f"{uri_of_the_product}/submodels/digital-nameplate",
            "idShort": "DigitalNameplate",
            "submodelElements": nameplate_elements
        },
        {
            "id": f"{uri_of_the_product}/submodels/maintenance-instructions",
            "idShort": "MaintenanceInstructions",
            "submodelElements": maintenance_instructions_elements
        },
        {
            "id": f"{uri_of_the_product}/submodels/handover-documentation",
            "idShort": "HandoverDocumentation",
            "submodelElements": handover_documentation_elements
        },

    ]
}

# 4. Save the combined, ready-to-upload AAS environment file
output_filename = "final_basyx_aas_environment.json"
with open(output_filename, "w") as f:
    json.dump(aas_environment, f, indent=2)

print(f"Successfully generated {output_filename} linked to {uri_of_the_product}")

https://www.fibrort.com/downloads?product=25
Successfully generated final_basyx_aas_environment.json linked to https://www.fibrort.com/downloads?product=25


In [203]:
import requests

from basyx.aas import model
from basyx.aas.model import DictObjectStore
from basyx.aas.adapter import json as aas_json
from basyx.aas.adapter.aasx import (
    AASXWriter,
    DictSupplementaryFileContainer,
)

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

JSON_FILE = output_filename           # Your generated AAS Environment JSON
AASX_FILE = "finaltemplate.aasx"
IMAGE_PATH = "product.png"             # Optional
UPLOAD_URL = "http://localhost:8081/upload"

# ------------------------------------------------------------------
# Read the JSON Environment
# ------------------------------------------------------------------

object_store = DictObjectStore()

with open(JSON_FILE, "r", encoding="utf-8") as f:
    try:
        aas_json.read_aas_json_file(f)
    except Exception as e:
        print(f"Error reading AAS JSON file: {e}")
        exit(1)


print("Environment loaded.")

# ------------------------------------------------------------------
# Create file store
# ------------------------------------------------------------------

file_store = DictSupplementaryFileContainer()

# If you have supplementary files, uncomment:
#
# file_store.add_file(
#     "/aasx/files/product.png",
#     IMAGE_PATH
# )

# ------------------------------------------------------------------
# Find all AAS IDs
# ------------------------------------------------------------------

aas_ids = [
    obj.id
    for obj in object_store
    if isinstance(obj, model.AssetAdministrationShell)
]

print("Found AAS IDs:")
for aid in aas_ids:
    print(" -", aid)

# ------------------------------------------------------------------
# Write the AASX package
# ------------------------------------------------------------------

with AASXWriter(AASX_FILE) as writer:
    writer.write_aas(
        aas_ids=aas_ids,
        object_store=object_store,
        file_store=file_store,
    )

print(f"AASX created: {AASX_FILE}")

# ------------------------------------------------------------------
# Upload to BaSyx
# ------------------------------------------------------------------

headers = {
    "Accept": "application/json"
}

with open(AASX_FILE, "rb") as f:
    response = requests.post(
        UPLOAD_URL,
        files={"file": (AASX_FILE, f, "application/octet-stream")},
        headers=headers,
    )

print("Status:", response.status_code)
print(response.text)

if response.status_code in (200, 201):
    print("Upload successful!")
else:
    print("Upload failed.")

Error while trying to convert JSON object into Property: Object with attribute (name='type', value='SMT/ExampleValue') is already present in this set of objects (Constraint AASd-021) >>> {'category': 'PARAMETER', 'displayName': [{...}, {...}], 'idShort': 'ClassificationSystem', 'modelType': 'Property', 'qualifiers': [{...}, {...}, {...}, {...}, {...}], 'semanticId': {'keys': [...], 'type': 'ExternalReference'}, 'supplementalSemanticIds': [{...}], 'valueType': 'xs:string'}
Traceback (most recent call last):
  File "/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/.venv/lib/python3.7/site-packages/basyx/aas/adapter/json/json_deserialization.py", line 208, in object_hook
    return AAS_CLASS_PARSERS[model_type](dct)
  File "/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/.venv/lib/python3.7/site-packages/basyx/aas/adapter/json/json_deserialization.py", line 719, in _construct_property
    cls._amend_abstract_attributes(ret, dct)
  File "/Users/alexialozano/Documents/GitHub/IDTA-AAS

Expected a SubmodelElement in SubmodelElementCollection, but found {'category': 'PARAMETER', 'idShort': 'ClassificationSystem', 'displayName': [{'language': 'en', 'text': 'Classification system'}, {'language': 'de', 'text': 'Klassifikationssystem'}], 'semanticId': {'type': 'ExternalReference', 'keys': [{'type': 'GlobalReference', 'value': '0173-1#02-ABL424#001'}]}, 'supplementalSemanticIds': [{'type': 'ExternalReference', 'keys': [{'type': 'GlobalReference', 'value': 'https://api.eclass-cdp.com/0173-1-02-ABL424-001'}]}], 'qualifiers': [{'semanticId': {'type': 'ExternalReference', 'keys': [{'type': 'GlobalReference', 'value': 'https://admin-shell.io/SubmodelTemplates/SMT/Cardinality/1/0'}]}, 'type': 'SMT/Cardinality', 'valueType': 'xs:string', 'value': 'One'}, {'semanticId': {'type': 'ExternalReference', 'keys': [{'type': 'GlobalReference', 'value': 'https://admin-shell.io/SubmodelTemplates/AllowedValue/1/0'}]}, 'type': 'SMT/ExampleValue', 'valueType': 'xs:string', 'value': 'ECLASS', 'v

Error reading AAS JSON file: Value is not a valid XSD date string
Environment loaded.
Found AAS IDs:
AASX created: finaltemplate.aasx
Status: 200
true
Upload successful!
